In [8]:
import sys
import subprocess

# Map pip package -> import name when they differ
pkgs = {
    'akshare': 'akshare',
    'futu-api': 'futu',
    'pandas': 'pandas',
    'tushare': 'tushare',
    'python-dotenv': 'dotenv',
    'IProgress': 'IProgress'
}

def ensure(pkg, module_name):
    try:
        __import__(module_name)
    except Exception:
        print(f'Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

for pkg, mod in pkgs.items():
    ensure(pkg, mod)

# Now import and load .env if present
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=False)
print('Environment and packages prepared.')

Environment and packages prepared.


In [9]:
from futu import *
quote_ctx = OpenQuoteContext(host='127.0.0.1', port=11111)
ret, data = quote_ctx.get_global_state()
if ret == RET_OK:
    print(data) # 检查其中的 'qutotation_permission' 字段
quote_ctx.close()

2026-04-13 21:46:14,041 | 32896 | 27144 | [open_context_base.py:434] _connect_sync: Connect fail: conn=0(1); msg=WSAECONNREFUSED; context=<futu.quote.open_quote_context.OpenQuoteContext object at 0x00000177E541FDF0>
2026-04-13 21:46:22,097 | 32896 | 27144 | [open_context_base.py:434] _connect_sync: Connect fail: conn=0(2); msg=WSAECONNREFUSED; context=<futu.quote.open_quote_context.OpenQuoteContext object at 0x00000177E541FDF0>
2026-04-13 21:46:30,127 | 32896 | 27144 | [open_context_base.py:434] _connect_sync: Connect fail: conn=0(3); msg=WSAECONNREFUSED; context=<futu.quote.open_quote_context.OpenQuoteContext object at 0x00000177E541FDF0>
2026-04-13 21:46:38,160 | 32896 | 27144 | [open_context_base.py:434] _connect_sync: Connect fail: conn=0(4); msg=WSAECONNREFUSED; context=<futu.quote.open_quote_context.OpenQuoteContext object at 0x00000177E541FDF0>
2026-04-13 21:46:46,207 | 32896 | 27144 | [open_context_base.py:434] _connect_sync: Connect fail: conn=0(5); msg=WSAECONNREFUSED; contex

In [10]:
from futu import *

# 1. 建立连接
quote_ctx = OpenQuoteContext(host='127.0.0.1', port=11111)

# 2. 设定 PDD 的 DCF 提醒逻辑
# 假设 dcf_target_price 是你计算出来的买入价
dcf_target_price = 66.5 

ret, data = quote_ctx.set_price_reminder(
    code='US.PDD', 
    op=SetPriceReminderOp.ADD, 
    reminder_type=PriceReminderType.PRICE_DOWN, # 价格跌到
    value=dcf_target_price, 
    note='DCF 24x 估值点',
    # 修正这里：设置为每日一次
    reminder_freq=PriceReminderFreq.ONCE_A_DAY 
)

if ret == RET_OK:
    # 注意：这里的 data 直接就是 ID 整数
    print(f"PDD 提醒设置成功！每日触发一次。ID: {data}")
else:
    # 如果失败，这里的 data 通常是错误字符串
    print(f"设置失败，原因: {data}")

# 3. 关闭连接
quote_ctx.close()

2026-04-13 21:46:52,246 | 32896 | 27144 | [open_context_base.py:409] _init_connect_sync: New connect ready: conn=7449679546527207025(7) context=<futu.quote.open_quote_context.OpenQuoteContext object at 0x00000177E541E470>
PDD 提醒设置成功！每日触发一次。ID: 1776142009935048313
2026-04-13 21:46:52,338 | 32896 | 14056 | [open_context_base.py:516] on_disconnect: Disconnected: conn=0(7) reason=CallClose


In [11]:
import akshare as ak
import pandas as pd

def get_fcf_smart(symbol="600519"):
    print(f"正在分析 {symbol} 的财务数据...")
    full_symbol = f"sh{symbol}" if symbol.startswith("60") else f"sz{symbol}"
    
    # 获取数据
    df = ak.stock_financial_report_sina(stock=full_symbol, symbol="现金流量表")
    
    # 打印前 5 个列名，方便排错
    columns = df.columns.tolist()
    print(f"【Debug】成功获取数据，表头包含: {columns[:5]}...")

    # 1. 智能寻找日期列
    date_col = next((c for c in columns if "期" in c or "日" in c), None)
    
    # 2. 智能寻找经营现金流列
    ocf_col = next((c for c in columns if "经营活动" in c and "净额" in c), None)
    
    # 3. 智能寻找资本支出列
    capex_col = next((c for c in columns if "固定资产" in c and "支付" in c), None)

    if not all([date_col, ocf_col, capex_col]):
        print("【错误】未能匹配到所需科目，请检查上方打印的表头！")
        return None
        
    print(f"【匹配成功】日期列: {date_col} | OCF列: {ocf_col} | CapEx列: {capex_col}")

    # 数据清洗
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
    df_annual = df[df[date_col].dt.month == 12].copy()
    
    df_annual[ocf_col] = pd.to_numeric(df_annual[ocf_col], errors='coerce')
    df_annual[capex_col] = pd.to_numeric(df_annual[capex_col], errors='coerce')
    
    # 计算 FCF
    df_annual["FCF_亿元"] = (df_annual[ocf_col] - df_annual[capex_col]) / 1e8
    df_annual["OCF_亿元"] = df_annual[ocf_col] / 1e8
    df_annual["CapEx_亿元"] = df_annual[capex_col] / 1e8
    
    return df_annual[[date_col, "OCF_亿元", "CapEx_亿元", "FCF_亿元"]].head(10)

if __name__ == "__main__":
    data = get_fcf_smart("600519")
    if data is not None:
        print("\n自由现金流分析结果 (最近10年)：")
        # 重命名日期列使其更易读
        data.rename(columns={data.columns[0]: "年份"}, inplace=True)
        # 将日期格式化为 YYYY-MM-DD
        data["年份"] = data["年份"].dt.strftime('%Y-%m-%d')
        print(data.to_string(index=False))

正在分析 600519 的财务数据...
【Debug】成功获取数据，表头包含: ['报告日', '经营活动产生的现金流量', '销售商品、提供劳务收到的现金', '客户存款和同业存放款项净增加额', '向中央银行借款净增加额']...
【匹配成功】日期列: 报告日 | OCF列: 经营活动产生的现金流量净额 | CapEx列: 购建固定资产、无形资产和其他长期资产所支付的现金

自由现金流分析结果 (最近10年)：
        年份     OCF_亿元  CapEx_亿元     FCF_亿元
2024-12-31 924.636922 46.787121 877.849801
2023-12-31 665.932477 26.197559 639.734918
2022-12-31 366.985958 53.065464 313.920494
2021-12-31 640.286761 34.087845 606.198916
2020-12-31 516.690687 20.897695 495.792992
2019-12-31 452.106126 31.488647 420.617480
2018-12-31 413.852344 16.067502 397.784842
2017-12-31 221.530361 11.250172 210.280189
2016-12-31 374.512496 10.191781 364.320715
2015-12-31 174.363401 20.614705 153.748697


In [1]:
import akshare as ak
import pandas as pd
import IProgress


# A股数据查询示例 — 茅台 600519

分两部分：**akshare**（免费，无需注册）和 **tushare**（需要 token）。

## Part 1 — akshare（免费）

In [7]:
import akshare as ak

SYMBOL = "sh600519"  # 茅台

try:
    # 获取从 2006 年至今的数据 (约 20 年)
    print("正在获取 20 年历史数据，请稍候...")
    
    # 方案 A: 继续尝试腾讯接口 (如果腾讯不支持这么久，请看下方的方案 B)
    df_20y = ak.stock_zh_a_hist_tx(
        symbol=SYMBOL, 
        start_date="20060101", 
        end_date="20260413", 
        adjust="qfq"
    )

    # 验证数据量
    print(f"数据获取成功！")
    print(f"共获取到 {len(df_20y)} 个交易日的数据。")
    
    # 查看前 5 行 (最早的 2006 年数据)
    print("\n=== 最早的 5 条数据 (2006年) ===")
    print(df_20y.head())
    
    # 查看后 5 行 (最近的 2024/2025年数据)
    print("\n=== 最近的 5 条数据 ===")
    print(df_20y.tail())

    # 如果你想保存到 Excel 慢慢看
    # df_20y.to_excel("moutai_20_years.xlsx", index=False)

except Exception as e:
    print(f"获取失败: {e}")

正在获取 20 年历史数据，请稍候...


  0%|          | 0/21 [00:00<?, ?it/s]

数据获取成功！
共获取到 4856 个交易日的数据。

=== 最早的 5 条数据 (2006年) ===
         date    open   close    high     low    amount
0  2006-01-04 -276.83 -277.21 -276.83 -277.26  35507.13
1  2006-01-05 -277.23 -277.05 -276.86 -277.26  16992.30
2  2006-01-06 -277.03 -276.89 -276.81 -277.18  17162.30
3  2006-01-09 -276.85 -276.36 -276.19 -276.86  20770.80
4  2006-01-10 -276.29 -276.30 -276.10 -276.45  17456.82

=== 最近的 5 条数据 ===
            date     open    close     high      low   amount
4851  2026-04-07  1460.05  1440.02  1470.00  1435.05  24152.0
4852  2026-04-08  1460.00  1465.02  1469.08  1452.13  33836.0
4853  2026-04-09  1465.02  1460.49  1465.02  1447.50  20705.0
4854  2026-04-10  1459.14  1453.96  1459.71  1441.10  28866.0
4855  2026-04-13  1444.00  1443.31  1446.50  1433.00  25214.0


In [ ]:
import requests, re, json, pandas as pd

# ── 日K线：新浪主力 / 腾讯兜底 ──────────────────────────────────
# 腾讯/新浪 raw 列顺序都是: date, open, close, high, low, amount
# 统一输出: date, open, high, low, close, amount

def _clean(df: pd.DataFrame) -> pd.DataFrame:
    """统一列顺序、类型、排序"""
    df = df.copy()
    df["date"]   = pd.to_datetime(df["date"])
    for c in ["open", "high", "low", "close", "amount"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.drop_duplicates("date").sort_values("date").reset_index(drop=True)
    return df[["date", "open", "high", "low", "close", "amount"]]


def fetch_sina(symbol: str, start: str, end: str, adjust: str = "qfq") -> pd.DataFrame:
    """
    新浪财经前/后复权日K。
    symbol: 带市场前缀，如 'sh600519'
    adjust: 'qfq' | 'hfq' | 'nofq'
    数据格式每行: date,open,close,high,low,volume,amount
    """
    url = f"https://finance.sina.com.cn/realstock/company/{symbol}/{adjust}.js"
    r = requests.get(url, headers={"User-Agent": "Mozilla/5.0", "Referer": "https://finance.sina.com.cn/"}, timeout=15)
    r.raise_for_status()

    # 提取 JS 里的 JSON 对象
    m = re.search(r'=(\{.*\})', r.text, re.DOTALL)
    if not m:
        raise ValueError(f"无法解析新浪返回: {r.text[:200]}")
    rows = json.loads(m.group(1)).get("d", [])

    records = []
    for line in rows:
        p = line.split(",")
        if len(p) < 6:
            continue
        records.append({
            "date":   p[0],
            "open":   p[1],
            "close":  p[2],
            "high":   p[3],
            "low":    p[4],
            "amount": p[6] if len(p) > 6 else p[5],
        })

    df = _clean(pd.DataFrame(records))
    return df[(df["date"] >= start) & (df["date"] <= end)].reset_index(drop=True)


def fetch_tx(symbol: str, start: str, end: str, adjust: str = "qfq") -> pd.DataFrame:
    """腾讯财经日K（via akshare），列映射同上"""
    raw = ak.stock_zh_a_hist_tx(
        symbol=symbol,
        start_date=start.replace("-", ""),
        end_date=end.replace("-", ""),
        adjust=adjust,
    )
    # akshare 已返回标准列: date, open, close, high, low, amount
    raw = raw.rename(columns={"amount": "amount"})  # 保持一致
    return _clean(raw)


# ── 主调用 ────────────────────────────────────────────────────
SYMBOL_MKT = "sh600519"   # 带市场前缀
START, END  = "2006-01-01", "2026-04-13"

try:
    kline = fetch_sina(SYMBOL_MKT, START, END, adjust="qfq")
    print(f"【新浪财经】获取成功，共 {len(kline)} 个交易日")
except Exception as e:
    print(f"【新浪失败】{e}\n【切换腾讯财经...】")
    kline = fetch_tx(SYMBOL_MKT, START, END, adjust="qfq")
    print(f"【腾讯财经】获取成功，共 {len(kline)} 个交易日")

print("\n=== 最早5条 ===")
print(kline.head())
print("\n=== 最近5条 ===")
print(kline.tail())

In [16]:
# ── 3. 利润表（新浪财经）──────────────────────────────────────
# symbol 可选: "利润表" | "资产负债表" | "现金流量表"
profit = ak.stock_financial_report_sina(stock=f"sh{SYMBOL}", symbol="利润表")
print("=== 利润表列名 ===")
print(profit.columns.tolist()[:10])
print()
# 取关键指标
cols_want = ["报告日","营业总收入","营业利润","净利润"]
available = [c for c in cols_want if c in profit.columns]
print(profit[available].head(6).to_string(index=False))

=== 利润表列名 ===
['报告日', '营业总收入', '营业收入', '利息收入', '已赚保费', '手续费及佣金收入', '房地产销售收入', '其他业务收入', '营业总成本', '营业成本']

     报告日                营业总收入                 营业利润                 净利润
20250930  130903889634.880005   89489694566.479996  66898804746.169998
20250630   91093762553.970001   62768973374.370003  46986681449.239998
20250331   51443450583.769997        37036598228.0  27774636011.610001
20241231  174144069958.249969  119688579453.229996  89334728025.899994
20240930  123122542625.449997   83996733560.979996  63031462239.550003
20240630   83451164646.529999   57550951619.010002  43176914345.120003


In [17]:
# ── 4. 资产负债表 ─────────────────────────────────────────────
balance = ak.stock_financial_report_sina(stock=f"sh{SYMBOL}", symbol="资产负债表")
cols_want = ["报告日","货币资金","应收账款","总资产","总负债","股东权益合计"]
available = [c for c in cols_want if c in balance.columns]
print("=== 资产负债表（精简）===")
print(balance[available].head(6).to_string(index=False))

=== 资产负债表（精简）===
     报告日                货币资金         应收账款
20250930  51753057846.449997  25531737.62
20250630      51644880716.93  37961242.39
20250331  52199240419.269997  15199312.09
20241231  59295822956.889999  18974192.75
20240930  60084745614.269997  10131702.74
20240630      56840349530.82   1358595.12


In [18]:
# ── 5. 股票基本信息（东方财富）────────────────────────────────
info = ak.stock_individual_info_em(symbol=SYMBOL)
# 返回 DataFrame with columns: item, value
print("=== 个股基本信息 ===")
print(info.to_string(index=False))

=== 个股基本信息 ===
item                 value
  最新               1439.26
股票代码                600519
股票简称                  贵州茅台
 总股本          1252270215.0
 流通股          1252270215.0
 总市值  1802342429640.899902
流通市值  1802342429640.899902
  行业                   白酒Ⅱ
上市时间              20010827


In [19]:
# ── 6. 分红派息历史 ───────────────────────────────────────────
dividend = ak.stock_divident_cninfo(symbol=SYMBOL)
print("=== 历史分红（最近5次）===")
print(dividend.head())

AttributeError: module 'akshare' has no attribute 'stock_divident_cninfo'

In [20]:
# ── 7. 机构持股（龙虎榜 / 主力资金）─────────────────────────
# 每日主力净流入（东方财富）
flow = ak.stock_individual_fund_flow(stock=SYMBOL, market="sh")
print("=== 主力资金流向（最近5天）===")
print(flow.tail())

=== 主力资金流向（最近5天）===
             日期      收盘价   涨跌幅     主力净流入-净额  主力净流入-净占比    超大单净流入-净额  \
115  2026-04-07  1440.02 -1.37 -340527856.0      -9.71 -272269584.0   
116  2026-04-08  1465.02  1.74  201450816.0       4.07  -17807024.0   
117  2026-04-09  1460.49 -0.31 -241092656.0      -8.00  -52333824.0   
118  2026-04-10  1453.96 -0.45 -163078000.0      -3.89  -27641152.0   
119  2026-04-13  1443.31 -0.73 -250575824.0      -6.90  -50204624.0   

     超大单净流入-净占比     大单净流入-净额  大单净流入-净占比     中单净流入-净额  中单净流入-净占比  小单净流入-净额  \
115       -7.76  -68258272.0      -1.95  340700704.0       9.71 -172855.0   
116       -0.36  219257840.0       4.43 -201184608.0      -4.07 -266205.0   
117       -1.74 -188758832.0      -6.27  241156800.0       8.01  -64149.0   
118       -0.66 -135436848.0      -3.23  163231536.0       3.90 -153525.0   
119       -1.38 -200371200.0      -5.52  250591664.0       6.90  -15839.0   

     小单净流入-净占比  
115      -0.00  
116      -0.01  
117      -0.00  
118      -0.00  
119  

## Part 2 — tushare（需要 token）

先在 `.env` 里设置 `TUSHARE_TOKEN=xxx`，或直接 `ts.set_token("xxx")`。

In [ ]:
import tushare as ts
import os
from dotenv import load_dotenv

load_dotenv()  # 读取 .env 文件
ts.set_token(os.getenv("TUSHARE_TOKEN", ""))
pro = ts.pro_api()

TS_CODE = "600519.SH"  # tushare 格式：代码.交易所

In [ ]:
# ── 1. 股票基本信息 ───────────────────────────────────────────
basic = pro.stock_basic(ts_code=TS_CODE,
                        fields="ts_code,name,area,industry,list_date,market")
print("=== 基本信息 ===")
print(basic.to_string(index=False))

In [ ]:
# ── 2. 日K线（后复权）─────────────────────────────────────────
# adj: None 不复权 | "qfq" 前复权 | "hfq" 后复权
daily = pro.daily(ts_code=TS_CODE, start_date="20240101", end_date="20241231")
print("=== 日K线（最近5条）===")
print(daily.head())

In [ ]:
# ── 3. 复权因子（用来自己算复权价）─────────────────────────
adj_factor = pro.adj_factor(ts_code=TS_CODE, start_date="20240101")
print("=== 复权因子（最近5条）===")
print(adj_factor.head())

In [ ]:
# ── 4. 每日基本面指标（PE/PB/市值等）────────────────────────
daily_basic = pro.daily_basic(
    ts_code=TS_CODE,
    start_date="20240101",
    end_date="20241231",
    fields="trade_date,pe,pe_ttm,pb,ps_ttm,total_mv,circ_mv,turnover_rate",
)
print("=== 每日基本面（最近5条）===")
print(daily_basic.head())

In [ ]:
# ── 5. 利润表（年报）─────────────────────────────────────────
# report_type: 1=合并报表 | period: 年报末尾是 1231
income = pro.income(
    ts_code=TS_CODE,
    period="20231231",
    report_type="1",
    fields="ts_code,end_date,total_revenue,operate_profit,n_income_attr_p",
)
print("=== 年度利润表 ===")
print(income.to_string(index=False))

In [ ]:
# ── 6. 资产负债表 ─────────────────────────────────────────────
balance = pro.balancesheet(
    ts_code=TS_CODE,
    period="20231231",
    report_type="1",
    fields="ts_code,end_date,money_cap,total_assets,total_liab,total_hldr_eqy_exc_min_int",
)
print("=== 资产负债表 ===")
print(balance.to_string(index=False))

In [ ]:
# ── 7. 现金流量表（算 FCF）─────────────────────────────────
cashflow = pro.cashflow(
    ts_code=TS_CODE,
    period="20231231",
    report_type="1",
    fields="ts_code,end_date,n_cashflow_act,c_pay_acq_const_fiolta",
)
# n_cashflow_act       = 经营活动净现金流
# c_pay_acq_const_fiolta = 购建固定资产等支付的现金（CapEx）
cashflow["FCF"] = cashflow["n_cashflow_act"] - cashflow["c_pay_acq_const_fiolta"]
print("=== 现金流量表 + FCF ===")
print(cashflow.to_string(index=False))

In [ ]:
# ── 8. 分红送股记录 ───────────────────────────────────────────
dividend = pro.dividend(ts_code=TS_CODE, fields="ts_code,end_date,cash_div_tax,stk_div")
print("=== 历史分红（最近5条）===")
print(dividend.head())

In [ ]:
# ── 9. 主要财务指标（ROE / EPS / 每股FCF 等）──────────────
fina = pro.fina_indicator(
    ts_code=TS_CODE,
    period="20231231",
    fields="ts_code,end_date,roe,eps,bps,netprofit_margin,grossprofit_margin,fcff",
)
# fcff = 企业自由现金流量（FCFF），单位：元/股（部分版本可能是总额，视 token 权限而定）
print("=== 主要财务指标 ===")
print(fina.to_string(index=False))

In [3]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv


def _fmp_get(endpoint: str, api_key: str, params: dict | None = None, timeout: int = 30):
    """Call FMP v3 endpoint and return parsed JSON with useful error details."""
    query = dict(params or {})
    query["apikey"] = api_key
    url = f"https://financialmodelingprep.com/api/v3/{endpoint}"

    try:
        resp = requests.get(url, params=query, timeout=timeout)
        resp.raise_for_status()
    except requests.exceptions.HTTPError:
        body_preview = ""
        try:
            body_preview = (resp.text or "")[:500]
        except Exception:
            body_preview = "<无法读取响应体>"
        raise RuntimeError(
            f"FMP HTTP错误: status={getattr(resp, 'status_code', 'unknown')} url={getattr(resp, 'url', url)} "
            f"response={body_preview}"
        )
    except requests.exceptions.RequestException as e:
        raise RuntimeError(f"FMP网络错误: {e} | url={url}")

    try:
        data = resp.json()
    except ValueError:
        raise RuntimeError(f"FMP返回非JSON内容 | url={getattr(resp, 'url', url)}")

    if isinstance(data, dict) and data.get("Error Message"):
        raise RuntimeError(f"FMP业务错误: {data['Error Message']} | url={resp.url}")
    if isinstance(data, dict) and data.get("error"):
        raise RuntimeError(f"FMP返回error: {data['error']} | url={resp.url}")

    return data


def _to_sorted_df(raw_rows):
    df = pd.DataFrame(raw_rows)
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        df = df.sort_values("date").reset_index(drop=True)
    return df


def _safe_pick(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    existing = [c for c in cols if c in df.columns]
    return df[existing].copy()


def get_fmp_financials_30y(ticker: str = "AAPL", limit_years: int = 30):
    """
    Fetch annual financial statements from FMP (income/balance/cashflow),
    plus a merged key-metrics table.
    """
    load_dotenv()
    api_key = os.getenv("FMP_API_KEY")
    if not api_key:
        raise ValueError("未读取到 FMP_API_KEY，请检查项目根目录 .env")

    income_raw = _fmp_get(f"income-statement/{ticker}", api_key, {"limit": limit_years})
    balance_raw = _fmp_get(f"balance-sheet-statement/{ticker}", api_key, {"limit": limit_years})
    cashflow_raw = _fmp_get(f"cash-flow-statement/{ticker}", api_key, {"limit": limit_years})

    df_income = _to_sorted_df(income_raw)
    df_balance = _to_sorted_df(balance_raw)
    df_cashflow = _to_sorted_df(cashflow_raw)

    income_cols = [
        "date", "calendarYear", "reportedCurrency", "revenue", "grossProfit",
        "operatingIncome", "netIncome", "eps"
    ]
    balance_cols = [
        "date", "calendarYear", "totalAssets", "totalLiabilities",
        "totalStockholdersEquity", "cashAndCashEquivalents"
    ]
    cashflow_cols = [
        "date", "calendarYear", "operatingCashFlow", "capitalExpenditure", "freeCashFlow"
    ]

    k_income = _safe_pick(df_income, income_cols)
    k_balance = _safe_pick(df_balance, balance_cols)
    k_cashflow = _safe_pick(df_cashflow, cashflow_cols)

    df_key = (
        k_income
        .merge(k_balance, on=["date", "calendarYear"], how="outer")
        .merge(k_cashflow, on=["date", "calendarYear"], how="outer")
        .sort_values("date")
        .reset_index(drop=True)
    )

    return {
        "income": df_income,
        "balance": df_balance,
        "cashflow": df_cashflow,
        "key": df_key,
    }


# ===== 示例：拉取 AAPL 过去30年并导出 =====
financials = get_fmp_financials_30y("AAPL", 30)

df_income = financials["income"]
df_balance = financials["balance"]
df_cashflow = financials["cashflow"]
df_key = financials["key"]

print("income rows:", len(df_income))
print("balance rows:", len(df_balance))
print("cashflow rows:", len(df_cashflow))

display(df_key.tail(10))

# 导出为 CSV（保存到当前工作目录）
df_income.to_csv("AAPL_income_30y.csv", index=False)
df_balance.to_csv("AAPL_balance_30y.csv", index=False)
df_cashflow.to_csv("AAPL_cashflow_30y.csv", index=False)
df_key.to_csv("AAPL_key_financials_30y.csv", index=False)

print("导出完成: AAPL_income_30y.csv / AAPL_balance_30y.csv / AAPL_cashflow_30y.csv / AAPL_key_financials_30y.csv")

RuntimeError: FMP HTTP错误: status=403 url=https://financialmodelingprep.com/api/v3/income-statement/AAPL?limit=30&apikey=Y2OW5BnKtzWPmQrMXlRT9zbiehHiS1Ik response={
  "Error Message": "Legacy Endpoint : Due to Legacy endpoints being no longer supported - This endpoint is only available for legacy users who have valid subscriptions prior August 31, 2025. Please visit our documentation page https://site.financialmodelingprep.com/developer/docs for our current APIs. "
}